# Workshop 3: Solutions

| Minutes | Activity |
|---|---|
| 0–2 | Getting started |
| 2–10 | Part 1: Solving versus training |
| 10–30 | Part 2: What is a neural network? |
| 30–35 | Part 3: How a network learns |
| 35–36 | Lab check |
| 36–41 | Demo: all 8 inputs |

## Getting started (2 minutes)

- Save your own copy in Colab (**File → Save a copy in Drive**).
- Click the **play button** beside a code cell to run it. Work from top to bottom.
- Double-click text to edit an answer. Click outside when done.

The first cell loads the libraries from Workshops 1 and 2, then the [California house prices from Workshop 1](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#getting-started-2-minutes) into a table called `housing`.

This session introduces a new library, **PyTorch**, which provides the building blocks of a neural network.

In [ ]:
# Libraries from Workshops 1 and 2.
import pandas as pd  # Stores data in tables.
import numpy as np  # Does maths on many numbers at once.
import matplotlib.pyplot as plt  # Draws graphs.
from sklearn.datasets import fetch_california_housing  # The house prices from Workshop 1.
from sklearn.linear_model import LinearRegression  # The straight line from Workshop 1.
from sklearn.model_selection import train_test_split  # Splits the data, as in Workshop 1.
from sklearn.preprocessing import StandardScaler  # Rescales inputs, as in Workshop 2.

# New this week: PyTorch.
import torch  # Tensors and automatic gradients.
from torch import nn  # Building blocks for neural networks.

housing = fetch_california_housing(as_frame=True).frame  # Download the data as a table.
housing.head()  # Show the first five rows.

## Part 1: Solving versus training (8 minutes)

In [Workshop 1](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#2-fit-a-model-and-predict-5-minutes) you fitted a straight line:

$$
\text{value} = \text{slope} \times \text{income} + \text{intercept}
$$

Run the cell to see it again. Expect a slope of about **0.42** and an intercept of about **0.45**.

In [ ]:
line = LinearRegression()
line.fit(housing[["MedInc"]], housing["MedHouseVal"])
print("slope:", line.coef_[0], "| intercept:", line.intercept_)

scikit-learn can find the best line in one go because a straight line is a special case. Its two parameters each just get multiplied by the data, so the best values come from solving two simple equations.

In a neural network, thousands of parameters get multiplied by each other, layer after layer, and equations like that cannot be rearranged into an answer. So instead of solving for a network's parameters, we **train** them. This means starting from random parameters and improving them in small steps, thousands of times. PyTorch gives you the model and the tools, and leaves the rest to you.

The cell below is our own `fit`. It takes any PyTorch model and improves it by repeating four steps: predict, measure the error, work out which way to nudge every parameter, nudge. There is a lot going on here. Run it for now, and we'll explain what's happening later!

In [ ]:
def fit(model, inputs, targets):
    loss_fn = nn.MSELoss()  # The loss from Workshop 1.
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Makes the improvements.
    for step in range(1000):
        optimizer.zero_grad()  # Forget the last step.
        predicted = model(inputs)  # Predict a value for every house.
        loss = loss_fn(predicted, targets)  # Measure how wrong the predictions are.
        loss.backward()  # Work out which way to nudge every parameter.
        optimizer.step()  # Nudge them.
    return loss.item()

We can build the same model in PyTorch, because linear regression is the simplest possible neural network. `nn.Linear(1, 1)` has one input and one output, and like `LinearRegression` it has a slope and an intercept, which PyTorch calls `weight` and `bias`. `incomes` and `values` are the two columns in PyTorch's format. The cell below prints the slope and intercept before and after training.

In [ ]:
incomes = torch.tensor(housing[["MedInc"]].to_numpy(), dtype=torch.float32)
values = torch.tensor(housing[["MedHouseVal"]].to_numpy(), dtype=torch.float32)

line = nn.Linear(1, 1)
print("before | slope:", line.weight.item(), "| intercept:", line.bias.item())
fit(line, incomes, values)
print("after  | slope:", line.weight.item(), "| intercept:", line.bias.item())

Side by side:

| | scikit-learn | PyTorch |
|---|---|---|
| Model | `LinearRegression()` | `nn.Linear(1, 1)` |
| Fitting | `line.fit(...)` solves | `fit(line, ...)` trains |
| Parameters | `coef_`, `intercept_` | `weight`, `bias` |

The one difference that matters is that scikit-learn's `fit` only works for its own models, and ours works for any PyTorch model. Swap in `nn.Linear(5, 10)`, which takes 5 inputs and gives 10 outputs, and the same `fit` trains it. We now have a recipe for training any neural network we can imagine!

## Part 2: What is a neural network? (20 minutes)

So far the only thing we have trained is a straight line. This part turns it into a neural network, one piece at a time. Each step has a diagram to play with and shows how to write the same thing in PyTorch.

Some diagrams in this part are interactive cells: run each one and use its controls. The code that draws them is hidden: click **Show code** to see it. The rest are pictures from the workshop page.

### 1. Neurons

#### Theory

A **neuron** takes some inputs. It multiplies each one by its own **weight**, adds them up, adds a **bias**, then passes the result through an **activation function**. The weights and bias are the neuron's parameters.

$$
\text{output} = \text{activation}(w_1 \times \text{input}_1 + w_2 \times \text{input}_2 + w_3 \times \text{input}_3 + b)
$$

![A neuron with three inputs: each input times its weight, plus the bias, added up, then the activation.](https://raw.githubusercontent.com/WarwickAI/intro-to-ml/main/workshop3/images/neuron.png)

Reading left to right, each input is multiplied by its weight on the way in, the bias joins from above, everything is added up, and the activation function decides what comes out.

Let's think about what the blue part is doing intuitively. Each input is some feature we are trying to reason about, like a house's income or latitude. Each weight says how much that feature matters to this neuron, and which way. Adding the weighted features to the bias gives us the blue part's output, which we'll call $z$.

$$
z = w_1 \times \text{input}_1 + w_2 \times \text{input}_2 + w_3 \times \text{input}_3 + b
$$

Notice that $z$ is linear in its features. If we built a network out of neurons with just the blue part, the whole thing would be linear too, however many neurons we used. That's what the green part is for. The activation function makes each neuron non-linear, so a network of them can follow curves. There are several to choose from. We use **ReLU**, the most common one. It turns every negative value into 0 and leaves positive values alone, like a hinge.

$$
\text{output} = \text{ReLU}(z) = \max(0, z)
$$

With one input, the blue part is $w \times \text{input} + b$, which is Part 1's straight line, so a neuron is that line with ReLU on the end. Drag the sliders. The weight sets how steep the line is and which way it faces, and the bias moves the bend left and right.

$$
\text{output} = \text{ReLU}(w \times \text{input} + b)
$$

In [ ]:
# @title One neuron
from ipywidgets import FloatSlider, interact


@interact(weight=FloatSlider(min=-2, max=2, step=0.1, value=1), bias=FloatSlider(min=-3, max=3, step=0.1, value=0))
def one_neuron(weight, bias):
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

    left.set(xlim=(0, 10), ylim=(0, 4), title="output = ReLU(weight × x + bias)")
    left.axis("off")
    left.scatter([1.2, 5.2], [2, 2], s=[1500, 4000], color=["white", "tab:blue"], edgecolors="black", zorder=2)
    left.text(1.2, 2, "x", ha="center", va="center", fontsize=16)
    left.text(5.2, 2, f"{'+' if bias >= 0 else '−'} {abs(bias):.1f}\nReLU", ha="center", va="center", fontsize=12, color="white")
    left.annotate("", xy=(4.1, 2), xytext=(1.9, 2), arrowprops={"arrowstyle": "->", "linewidth": 2})
    left.text(3, 2.25, f"× {weight:.1f}".replace("-", "−"), ha="center", fontsize=12)
    left.annotate("", xy=(8.4, 2), xytext=(6.3, 2), arrowprops={"arrowstyle": "->", "linewidth": 2})
    left.text(8.5, 2, "output", va="center", fontsize=12)

    x = np.linspace(-5, 5, 201)
    line = weight * x + bias
    right.plot(x, line, "--", color="grey", label="weight × x + bias")
    right.plot(x, np.maximum(line, 0), color="tab:blue", linewidth=3, label="after ReLU")
    if weight != 0 and abs(bias / weight) <= 5:
        right.scatter([-bias / weight], [0], color="black", zorder=3, label="bend")
    right.axhline(0, color="black", linewidth=0.5)
    right.set(xlabel="x (input)", ylabel="output", ylim=(-5, 5))
    right.legend(loc="upper left")
    plt.show()

#### In PyTorch

PyTorch has a building block for each coloured part of the diagram. The blue part is `nn.Linear`. `nn.Linear(a, b)` takes $a$ numbers in and gives $b$ numbers out, so one neuron with three inputs needs 3 in and 1 out.

```python
nn.Linear(3, 1)  # 3 numbers in, 1 number out.
```

With one input it is `nn.Linear(1, 1)`, the model from Part 1. The green part is the activation.

```python
nn.ReLU()  # Turns negative values into 0.
```

`nn.Sequential` runs its pieces in order, so the output of the blue part goes into the green part.

In [ ]:
neuron = nn.Sequential(nn.Linear(3, 1), nn.ReLU())

Write the one-input neuron by hand using `torch.relu`, with a weight of 2 and a bias of −4, in one line. Expect `tensor([0., 0., 4., 8., 12., 16.])`.

In [ ]:
x = torch.tensor([0.0, 2.0, 4.0, 6.0, 8.0, 10.0])
torch.relu(2 * x - 4)  # One neuron: weight 2, bias -4.

Where does this neuron bend?

At $x = 2$, where $2x - 4 = 0$. Below 2 the neuron outputs 0. Above 2 its output grows by 2 for every 1 that $x$ grows, because its weight is 2.

Neurons are the building blocks of neural networks. Now we just need to combine them!

### 2. Layers

#### Theory

A **layer** is several neurons that all read the same inputs and give different outputs. Start with one neuron with three inputs. Put a second neuron beside it, reading the same three inputs with its own weights and bias, and you have two outputs. Keep going and you have a layer. Each neuron has its own weights and bias, so each one picks out something different from the same features.

![A layer of three neurons reading the same three inputs.](https://raw.githubusercontent.com/WarwickAI/intro-to-ml/main/workshop3/images/layer.png)

On the workshop page you can add and remove neurons and watch the code change.

#### In PyTorch

A layer is one `nn.Linear` with more outputs. PyTorch doesn't write two neurons as two separate `nn.Linear(3, 1)`s. It writes the whole layer as one piece, with one output per neuron.

```python
nn.Linear(3, 1)  # One neuron. 3 numbers in, 1 out.
nn.Linear(3, 2)  # Two neurons side by side. 3 in, 2 out.
nn.Linear(3, 3)  # Three neurons. 3 in, 3 out.
```

One `nn.ReLU()` bends every output at once, so a layer of three neurons is two pieces, the same as a single neuron.

In [ ]:
layer = nn.Sequential(nn.Linear(3, 3), nn.ReLU())

On the workshop page you can add and remove neurons and watch the code change.

### 3. Neural networks

#### Theory

A **neural network** is layers chained together, so that each layer's outputs are the next layer's inputs. The layers in the middle are **hidden layers**. The last layer is a single neuron with no activation, and its output is the prediction. There is no ReLU on the end because a prediction can be any number, negative ones included.

Here is a network with one input, two hidden layers of three neurons each, and one output. Each circle is a neuron, blue add up then green ReLU. The output neuron is all blue because it has no activation.

![A network with one input, two hidden layers of three neurons and one output.](https://raw.githubusercontent.com/WarwickAI/intro-to-ml/main/workshop3/images/network.png)

What does a network compute? To see, take one hidden layer. The three hidden neurons below are already set to fit the dots. Add them one at a time and watch the line change. Each numbered dot on the line marks the bend made by the neuron with that number.

In [ ]:
# @title Neurons add up
from IPython.display import Markdown, display
from ipywidgets import Checkbox, ToggleButtons, interact

dots_x = np.arange(0, 10.5, 0.5)
dots_y = 1 + np.maximum(dots_x - 2, 0) - 2 * np.maximum(dots_x - 5, 0) + np.maximum(dots_x - 8, 0)
neurons = [(1, -2, 1), (1, -5, -2), (1, -8, 1)]  # (weight, bias, output weight) for each neuron.
output_bias = 1
colours = ["tab:blue", "tab:orange", "tab:green"]
changes = ["starts to climb", "turns down", "flattens out"]
choices = [("No neurons", 0), ("1 neuron", 1), ("2 neurons", 2), ("3 neurons", 3)]


@interact(added=ToggleButtons(options=choices, description=""), relu=Checkbox(value=True, description="ReLU"))
def neurons_add_up(added, relu):
    def output(x):
        total = np.full(len(x), float(output_bias))
        for weight, bias, output_weight in neurons[:added]:
            line = weight * x + bias
            total = total + output_weight * (np.maximum(line, 0) if relu else line)
        return total

    if added == 0:
        display(Markdown(f"No neurons yet: only the output's bias, a flat line at ${output_bias}$."))
    else:
        weight, bias, output_weight = neurons[added - 1]
        if relu:
            display(Markdown(rf"Neuron {added} adds ${output_weight} \times \text{{ReLU}}(\text{{input}} - {-bias})$, so the line {changes[added - 1]} at $\text{{input}} = {-bias}$."))
        else:
            display(Markdown(rf"Neuron {added} adds ${output_weight} \times (\text{{input}} - {-bias})$, a straight line, so the output stays straight."))

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))

    left.set(xlim=(0, 10), ylim=(-0.5, 4.5), title="Input → hidden layer → output")
    left.axis("off")
    for n, ((weight, bias, output_weight), colour) in enumerate(zip(neurons, colours)):
        y = 3.5 - 1.5 * n
        alpha = 1 if n < added else 0.15
        label = f"ReLU(input − {-bias})" if relu else f"input − {-bias}"
        left.plot([1, 3.6], [2, y], color="grey", alpha=alpha, zorder=1)
        left.plot([6.4, 9], [y, 2], color=colour, linewidth=1 + abs(output_weight), alpha=alpha, zorder=1)
        left.text(5, y, label, ha="center", va="center", fontsize=10, color="white", alpha=alpha, zorder=2,
                  bbox={"boxstyle": "round,pad=0.5", "facecolor": colour, "edgecolor": "black", "alpha": alpha})
        left.text(3.3, y + 0.45, str(n + 1), ha="center", va="center", fontsize=9, alpha=alpha, zorder=3,
                  bbox={"boxstyle": "circle,pad=0.3", "facecolor": "white", "edgecolor": "black", "alpha": alpha})
        left.text(7.6, (y + 2) / 2 + 0.3, f"× {output_weight}".replace("-", "−"), ha="center", fontsize=10, alpha=alpha)
    left.scatter([1, 9], [2, 2], s=1600, color="white", edgecolors="black", zorder=2)
    left.text(1, 2, "input", ha="center", va="center", fontsize=11)
    left.text(9, 2, f"+ {output_bias}", ha="center", va="center", fontsize=10)

    x = np.linspace(0, 10, 201)
    right.scatter(dots_x, dots_y, color="grey", s=15, zorder=2)
    right.plot(x, output(x), color="black", linewidth=3, zorder=3)
    if relu:
        for n, ((weight, bias, output_weight), colour) in enumerate(zip(neurons[:added], colours)):
            bend = -bias / weight
            height = output(np.array([bend]))[0]
            right.scatter([bend], [height], s=120, color=colour, edgecolors="black", zorder=4)
            right.text(bend, height + 0.45, str(n + 1), ha="center", va="center", fontsize=9, zorder=5,
                       bbox={"boxstyle": "circle,pad=0.25", "facecolor": "white", "edgecolor": "black"})
    right.set(xlabel="input", xlim=(0, 10), ylim=(-1, 6), title="Prediction")
    plt.show()

1. What does each neuron add to the line?
2. Untick **ReLU** and add the neurons again. What happens?

1. Each neuron adds one bend, at the point where its $\text{weight} \times x + \text{bias}$ is 0: at $x = 2$, $5$ and $8$. Its output weight decides how much the line's slope changes there: $+1$ starts the climb, $-2$ turns it into a fall, and $+1$ levels it off. With all three, the line goes through every dot. Training finds weights and biases like these for you.
2. Without ReLU, each neuron is just a straight line, so adding one only tilts the output and never bends it. Straight lines always add up to a straight line, however many you use.

That is the claim from section 1 made real. Without the green part, a network of any size is one straight line. The bends are what let it follow curves.

#### In PyTorch

There is nothing new to learn here. A network is layers from section 2 chained with `nn.Sequential`, each one's output feeding the next, and one last `nn.Linear` with no `nn.ReLU()` after it for the output.

In [ ]:
network = nn.Sequential(
    nn.Linear(1, 3), nn.ReLU(),  # First hidden layer.
    nn.Linear(3, 3), nn.ReLU(),  # Second hidden layer. Reads the first's 3 outputs.
    nn.Linear(3, 1),  # Output layer. No ReLU, so the prediction can be any number.
)

The one-hidden-layer network from the graph above fits in one line of code: the output's bias, plus each hidden neuron times its output weight. Write it and check that you get the dots: `tensor([1., 1., 1., 2., 3., 4., 3., 2., 1., 1., 1.])`.

In [ ]:
x = torch.arange(0.0, 11.0)  # 0, 1, 2, ..., 10
1 + torch.relu(x - 2) - 2 * torch.relu(x - 5) + torch.relu(x - 8)  # The output's bias, plus each neuron times its output weight.

### 4. Zooming out

If the pictures got confusing, here is the good news. In PyTorch you never draw neurons. You choose a shape, how many inputs, how many hidden layers, how many neurons in each and how many outputs, and `nn.Sequential` does the rest. We predict one number, the house value, so our networks have one output, but a network can predict several things at once.

Drag the sliders and watch the code and the parameter count change. Every arrow is a weight and every neuron has a bias, and together they are the network's **parameters**, the numbers that training adjusts.

In [ ]:
# @title Layers and parameters
from IPython.display import Markdown, display
from ipywidgets import IntSlider, interact


@interact(inputs=IntSlider(min=1, max=8, value=1), hidden_layers=IntSlider(min=1, max=3, value=1), neurons=IntSlider(min=1, max=8, value=3), outputs=IntSlider(min=1, max=8, value=1))
def layers_and_parameters(inputs, hidden_layers, neurons, outputs):
    sizes = [inputs] + [neurons] * hidden_layers + [outputs]
    parameters = sum(before * after + after for before, after in zip(sizes, sizes[1:]))  # Weights, then biases.
    pieces = [f"nn.Linear({before}, {after})" for before, after in zip(sizes, sizes[1:])]
    code = "nn.Sequential(\n" + "".join(f"    {piece},{' nn.ReLU(),' if n < len(pieces) - 1 else ''}\n" for n, piece in enumerate(pieces)) + ")"
    display(Markdown(f"```python\n{code}\n```"))
    names = ["inputs"] + [f"hidden layer {n + 1}" for n in range(hidden_layers)] + ["outputs"]
    layers = [[(2 * column, (size - 1) / 2 - row) for row in range(size)] for column, size in enumerate(sizes)]

    fig, ax = plt.subplots(figsize=(11, 4.5))
    for before, after in zip(layers, layers[1:]):
        for x1, y1 in before:
            for x2, y2 in after:
                ax.plot([x1, x2], [y1, y2], color="grey", linewidth=0.7, zorder=1)
    for column, (layer, name) in enumerate(zip(layers, names)):
        colour = "tab:blue" if 0 < column < len(layers) - 1 else "white"
        ax.scatter([x for x, y in layer], [y for x, y in layer], s=250, color=colour, edgecolors="black", zorder=2)
        ax.text(2 * column, -5.6, name, ha="center")
    ax.set(xlim=(-1, 2 * len(sizes) - 1), ylim=(-6, 5), title=f"{parameters:,} parameters, every weight and bias added up")
    ax.axis("off")
    plt.show()

GPT-3 has 175 billion parameters, today's largest models have trillions, and all of them were trained with the same kind of loop as `fit`. It goes the other way too. A few thousand parameters are enough to fly a drone, and a few million to walk a humanoid robot.

PyTorch can count them for you. `example_network.parameters()` gives every weight and bias, and `p.numel()` counts the numbers in each one. Count the parameters of a network with 8 inputs and two hidden layers of 64 neurons. Expect **4801**.

In [ ]:
example_network = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
sum(p.numel() for p in example_network.parameters())  # Add up p.numel() for every p in example_network.parameters().

### 5. Another way to see it

Here is a second way to think about a network that only uses the building blocks of PyTorch.

`nn.Linear(a, b)` is a box that turns $a$ numbers into $b$ numbers. That's all.

A network is boxes in a row, and the only rule is that the counts have to match up. Whatever comes out of one box goes into the next, so the second number of one `nn.Linear` must equal the first number of the next.

In [ ]:
nn.Sequential(
    nn.Linear(8, 64),  # 8 numbers in, 64 out.
    nn.ReLU(),
    nn.Linear(64, 64),  # 64 in, 64 out. Starts with 64 because 64 just came out.
    nn.ReLU(),
    nn.Linear(64, 1),  # 64 in, 1 out. Ends with 1 because we predict one number.
)

Between the boxes is an activation, which bends the numbers on their way through. `nn.ReLU()` is the usual choice. Two others you will see are `nn.Sigmoid()`, which squashes every number into the range 0 to 1, and `nn.Tanh()`, which squashes into −1 to 1. They each have their uses. What matters is that there is one between each pair of boxes and none after the last.

![The ReLU, Sigmoid and Tanh activation functions.](https://raw.githubusercontent.com/WarwickAI/intro-to-ml/main/workshop3/images/activations.png)

So to build a network, write down the counts, chain the boxes, put a bend between each pair.

What goes wrong if the counts don't match, say `nn.Linear(8, 64)` followed by `nn.Linear(32, 1)`?

PyTorch refuses to run it. The first box hands over 64 numbers and the second expects 32, so you get

```
RuntimeError: mat1 and mat2 shapes cannot be multiplied (5x64 and 32x1)
```

where 5 is how many houses you passed in. If you see this error in the demo, a count is wrong somewhere.

### 6. Build one for real

House values go up and down with latitude, from San Diego in the south (33°) to the Oregon border (42°), so a straight line cannot follow them. The cell below rescales latitude, as in [Workshop 2](https://warwick.ai/education/courses/ml-ai-flagship/workshop-2#2-see-overfitting), and defines `draw`, which plots a model's predictions over the houses. Run it.

In [ ]:
latitude_scaler = StandardScaler()
latitudes = torch.tensor(latitude_scaler.fit_transform(housing[["Latitude"]].to_numpy()), dtype=torch.float32)
sample = housing.sample(2000, random_state=0)  # Plot 2,000 houses so the graph draws quickly.


def draw(model):
    grid = np.linspace(32.5, 42, 400).reshape(-1, 1)  # 400 latitudes from south to north.
    with torch.no_grad():  # Only predicting, not training.
        curve = model(torch.tensor(latitude_scaler.transform(grid), dtype=torch.float32))
    plt.scatter(sample["Latitude"], sample["MedHouseVal"], s=3, alpha=0.3)
    plt.plot(grid, curve, color="black")
    plt.xlabel("Latitude (degrees north)")
    plt.ylabel("Median house value ($100,000s)")
    plt.show()

Now the network from step 3 with 50 neurons instead of 3, trained on latitude with `fit` from Part 1. Expect a loss of about **1.11**. A straight line gets 1.30.

In [ ]:
model = nn.Sequential(
    nn.Linear(1, 50),  # 50 neurons, each with its own weight and bias.
    nn.ReLU(),  # Bend each one.
    nn.Linear(50, 1),  # Combine them into one prediction.
)
print("loss:", fit(model, latitudes, values))
draw(model)

Los Angeles is at about 34° and San Francisco at about 37.8°. What has the model found?

Two peaks, Los Angeles and the San Francisco Bay Area. Between them is a dip for the cheaper Central Valley, and values fall again further north. Nobody told the model where the cities are, or where to bend. It found both from the data.

Before you run it, predict: what happens if you delete the `nn.ReLU()` line? Then delete it and run the cell again.

A straight line, with a loss of 1.30, the same as a single `nn.Linear(1, 1)`. It is the untick from step 3 on real data. Two linear layers with nothing between them add up to one line, however many neurons are in the middle, so the 151 parameters do no more than 2.

In Workshops 1 and 2 you made lines bend by adding powers such as $x^2$ and $x^3$. You chose the powers, and the powers decided the shape. A network learns its shape instead. Training moves every weight and bias until the bends fit the data, and Part 3 shows how.

## Part 3: How a network learns (5 minutes)

Part 2 explains how to build a network but not how to train it. Training uses the `fit` function from Part 1, which we will now go through line by line. The cell builds a small network and trains it on income, so there is a model to talk about. Expect a loss of about **0.70**.

In [ ]:
def fit(model, inputs, targets):
    # Setup.
    loss_fn = nn.MSELoss()  # The loss from Workshop 1.
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Makes the improvements.

    # Loop.
    for step in range(1000):
        optimizer.zero_grad()  # Forget the last step.
        predicted = model(inputs)  # Predict a value for every house.
        loss = loss_fn(predicted, targets)  # Measure how wrong the predictions are.
        loss.backward()  # Work out which way to nudge every parameter.
        optimizer.step()  # Nudge them.
    return loss.item()


model = nn.Sequential(nn.Linear(1, 10), nn.ReLU(), nn.Linear(10, 1))  # A small network.
print("loss:", fit(model, incomes, values))

**Setup** creates two things needed for training. `loss_fn` is the mean squared error from Workshop 1, which scores a set of predictions against the true values. `optimizer` is handed every weight and bias in the model, through `model.parameters()`, along with a learning rate, and updates them.

**The loop** then repeats the same steps a thousand times. Each pass uses the model's current parameters. The first line, `optimizer.zero_grad()`, makes most sense last, so it comes last here.

- **`predicted = model(inputs)`** runs the network on every house at once and gives a predicted value for each one. On the first pass the parameters are random, so the predictions are nonsense.

- **`loss = loss_fn(predicted, targets)`** compares those predictions with the true values and gives one number, the loss. A big loss means bad parameters. It says how bad they are, but not which way to change each one.

- **`loss.backward()`** works out, for every parameter, which way to nudge it to make the loss smaller, and how much that would help. That is the parameter's **gradient**.

- **`optimizer.step()`** turns each gradient into an update. Every parameter moves a little, the other way from its gradient, so that the loss goes down.

  $$
  \text{slope} \leftarrow \text{slope} - \text{learning rate} \times \text{gradient}
  $$

  How big the update is depends on the optimiser. `SGD` just multiplies the gradient by the learning rate, as in the formula. `Adam`, which `fit` uses, also adapts the size of each parameter's steps as it goes.

- **`optimizer.zero_grad()`** clears the gradients before the next step. PyTorch adds new gradients onto old ones unless you tell it not to, so without this line every step would be pushed by all the steps before it.

**Picturing the whole loop.** Think of the loss as a landscape and the current parameters as a dot on it. With one parameter the landscape is a curve, with two it is a surface, and with thousands it is something you cannot draw, but the picture is the same. It is a **bowl**, with the best parameters at the bottom. The gradient is how steep the bowl is under the dot and which way is uphill, and each step moves the dot the other way. For a straight line it is exactly a bowl. For a network it is a bumpier bowl, but the idea holds.

The diagram shows one parameter, the slope of the income line from Part 1, with the intercept fixed at 0.45. The best slope, 0.42, sits at the bottom, and the orange arrow is one step. Click **Step** to take it, and keep clicking. Then click **Reset**, set the learning rate to 0.05 and try again. The dot overshoots the bottom and zigzags before it settles. At 0.06 it overshoots further every time.

In [ ]:
# @title Loss bowl
from IPython.display import display
from ipywidgets import Button, FloatSlider, HBox, Layout, Output, VBox

a, b, c = 18.59, -15.54, 3.95  # The loss is a × slope² + b × slope + c, with the intercept fixed at 0.45.
path = [0.0]
rate = FloatSlider(min=0.005, max=0.06, step=0.005, value=0.01, description="learning rate", readout_format=".3f",
                   style={"description_width": "initial"}, layout=Layout(width="350px"))
step_button = Button(description="Step")
reset_button = Button(description="Reset")
picture = Output()


def loss(slope):
    return a * slope ** 2 + b * slope + c


def gradient(slope):
    return 2 * a * slope + b


def draw(*_):
    slope = path[-1]
    following = slope - rate.value * gradient(slope)
    with picture:
        picture.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 3.8))
        slopes = np.linspace(-0.6, 1.4, 201)
        ax.plot(slopes, loss(slopes), color="grey")
        ax.scatter(path[:-1], [loss(s) for s in path[:-1]], color="tab:blue", alpha=0.35, s=20)
        if -0.6 <= slope <= 1.4:
            ax.annotate("", xy=(following, loss(following)), xytext=(slope, loss(slope)),
                        arrowprops={"arrowstyle": "->", "color": "tab:orange", "linewidth": 2})
            ax.scatter([slope], [loss(slope)], color="tab:blue", edgecolors="black", s=80, zorder=3)
            title = f"Step {len(path) - 1}: slope {slope:.2f}, loss {loss(slope):.2f}, gradient {gradient(slope):.1f}".replace("-", "−")
        else:
            title = "The dot has left the bowl: each step overshoots by more than the last."
        ax.set(xlim=(-0.6, 1.4), ylim=(0, 20), xlabel="slope (intercept fixed at 0.45)", ylabel="loss (MSE)", title=title)
        plt.show()


def step(_):
    if -0.6 <= path[-1] <= 1.4:
        path.append(path[-1] - rate.value * gradient(path[-1]))
        draw()


def reset(_):
    path[:] = [0.0]
    draw()


step_button.on_click(step)
reset_button.on_click(reset)
rate.observe(draw, names="value")
display(VBox([HBox([step_button, reset_button, rate]), picture]))
draw()

That is the whole of `fit`. Predict, measure, find the gradients, step, a thousand times over, for every weight and bias at once. Now try the pieces yourself.

Try finding a gradient by adding the backward pass. The cell below makes a line with a random slope and intercept, and measures its loss on the house prices. Expect a slope gradient of about **−15.2**.

In [ ]:
torch.manual_seed(0)  # Start from the same random line every run.
line = nn.Linear(1, 1)  # A straight line with a random slope and intercept.
loss = nn.MSELoss()(line(incomes), values)  # How wrong it is.
loss.backward()  # Find the gradient for every parameter.
print("slope:", line.weight.item(), "| slope gradient:", line.weight.grad.item())

Gradients add up. The cell below runs the backward pass twice on the same line, with nothing cleared in between. Predict what it prints, then run it. Then add one line at the top of the loop to fix it.

In [ ]:
torch.manual_seed(0)  # The same random line as above.
line = nn.Linear(1, 1)
for _ in range(2):  # Two backward passes.
    loss = nn.MSELoss()(line(incomes), values)
    loss.backward()
print("slope gradient:", line.weight.grad.item())

About **−30.3**, double the gradient from before, because the second backward pass added its gradient onto the first. `line.zero_grad()` at the top of the loop clears it, and the cell prints −15.2 again. That is what `optimizer.zero_grad()` does in `fit`, for every parameter the optimiser holds.

```python
torch.manual_seed(0)  # The same random line as above.
line = nn.Linear(1, 1)
for _ in range(2):  # Two backward passes.
    line.zero_grad()  # Clear the last gradient first.
    loss = nn.MSELoss()(line(incomes), values)
    loss.backward()
print("slope gradient:", line.weight.grad.item())
```

The cell below is `fit` with `SGD` in place of `Adam`, so the learning rate is the only setting. Run it, then change `learning_rate` to 0.1 and run it again.

In [ ]:
learning_rate = 0.01  # Then try 0.1.

line = nn.Linear(1, 1)
optimizer = torch.optim.SGD(line.parameters(), lr=learning_rate)
for step in range(101):
    optimizer.zero_grad()
    loss = nn.MSELoss()(line(incomes), values)
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print("step", step, "| loss", loss.item())

What happens to the loss with a learning rate of 0.1? Why?

At 0.01 the loss falls every step, heading for the 0.70 that Part 1 reached. At 0.1 it blows up, growing every step until it prints `inf` and then `nan`. The steps are too big, so each one overshoots the bottom of the bowl by more than the last, like 0.06 in the diagram.

## Lab check (1 minute)

Explain in one sentence each:

- Why a network needs ReLU between its layers.
- What one step of `fit` does.

- Without ReLU, every layer is a straight line, and lines added up are still a line, so extra layers add nothing. ReLU adds the bends.
- It predicts a value for every house, measures the loss, works out the gradient for every parameter with `loss.backward()`, and moves each parameter a small step downhill.

## Demo: all 8 inputs (5 minutes)

Now the task from Workshop 1: predict house values from all 8 inputs. The cell below makes [Workshop 1's split](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#t1-split-the-data-three-ways-3-minutes), so the scores compare, and rescales the inputs as in Workshop 2. Run it.

In [ ]:
train, rest = train_test_split(housing, test_size=0.3, random_state=0)  # The split from Workshop 1.
val, test = train_test_split(rest, test_size=0.5, random_state=0)  # The test set stays aside.
input_columns = housing.columns.drop("MedHouseVal")  # The 8 input columns.

scaler = StandardScaler()  # Learns each column's average and spread from the training set only.
X_train = torch.tensor(scaler.fit_transform(train[input_columns]), dtype=torch.float32)
X_val = torch.tensor(scaler.transform(val[input_columns]), dtype=torch.float32)
y_train = torch.tensor(train[["MedHouseVal"]].to_numpy(), dtype=torch.float32)
y_val = torch.tensor(val[["MedHouseVal"]].to_numpy(), dtype=torch.float32)

Then the 8-64-64-1 network whose parameters you counted in Part 2, trained with `fit` and scored on the validation set. We build it live: fill in the five layers. Expect a validation MSE of about **0.3**.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(8, 64),  # 8 inputs into 64 neurons.
    nn.ReLU(),  # Bend them.
    nn.Linear(64, 64),  # A second hidden layer of 64 neurons.
    nn.ReLU(),  # Bend them.
    nn.Linear(64, 1),  # One prediction.
)
print("training MSE:", fit(model, X_train, y_train))
with torch.no_grad():  # Only measuring, not training.
    print("validation MSE:", nn.MSELoss()(model(X_val), y_val).item())

Compare it with [Workshop 1's models](https://warwick.ai/education/courses/ml-ai-flagship/workshop-1#take-home-find-the-sweet-spot-about-15-minutes), all scored on the same validation set. (Workshop 1's 0.52 for linear regression was scored on the data it trained on.)

| Model | Validation MSE |
|---|---|
| Always predict the average | 1.33 |
| Linear regression, 8 inputs | 0.54 |
| Decision tree, depth 9 | 0.43 |
| Random forest | 0.28 |
| This network | about 0.3 |

Five lines of PyTorch, trained with the same `fit` as the straight line in Part 1, land close to the random forest. The exact number depends on the random start and even on the machine: anywhere from about 0.28 to 0.34 is normal.